# Vincent Straussberger

## Fine-Tune Model

This notebook fine-tunes `Qwen/Qwen3-0.6B` on a separate RIS-based training set and then runs the tuned model on the Austrian tax benchmark.

Key facts:
- base model: `Qwen/Qwen3-0.6B`
- training input: `ris_tax_training_100.csv`
- benchmark input: `dataset_clean.csv`
- output: one submission CSV with `id,answer`


## How the code works

- loads the training CSV and blocks the benchmark from being used for training
- fine-tunes the model with LoRA and saves a local checkpoint
- runs the tuned model on the benchmark questions
- saves the final CSV in this folder


In [3]:
print('Skip this cell if your packages are already installed.')
print('Minimal install for this notebook: python -m pip install pandas numpy torch transformers sentence-transformers beautifulsoup4 pypdf requests tqdm')


Skip this cell if your packages are already installed.
Minimal install for this notebook: python -m pip install pandas numpy torch transformers sentence-transformers beautifulsoup4 pypdf requests tqdm


In [4]:
import csv
import json
import os
import random
import re
import statistics
import time
import warnings
from pathlib import Path

import pandas as pd
import requests
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

warnings.filterwarnings('ignore')
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


d:\DS Applications\.venv-gpu312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.11.0+cu128
CUDA available: True


In [5]:
def ensure_directory(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

CURRENT_DIR = Path.cwd().resolve()
REMOTE_DATASET_URL = 'https://raw.githubusercontent.com/svakulenk0/wu-llms-ss26/main/dataset_clean.csv'


def resolve_code_dir(current_dir: Path) -> Path:
    if current_dir.name == 'code':
        return current_dir
    candidate = current_dir / 'code'
    if candidate.exists():
        return candidate
    return current_dir


def resolve_submission_root(code_dir: Path) -> Path:
    if code_dir.name == 'code':
        return code_dir.parent
    if (code_dir / 'code').exists():
        return code_dir
    return code_dir.parent


def ensure_local_dataset(submission_root: Path, remote_url: str, filename: str = 'dataset_clean.csv') -> Path:
    local_path = submission_root / filename
    if local_path.exists():
        return local_path
    try:
        response = requests.get(remote_url, timeout=30)
        response.raise_for_status()
        local_path.write_text(response.text, encoding='utf-8')
        print(f'Downloaded benchmark dataset to {local_path}')
        return local_path
    except Exception as exc:
        raise FileNotFoundError(
            f'Could not find {filename} in the submission root and the download also failed. Original error: {exc}'
        ) from exc


NOTEBOOK_DIR = resolve_code_dir(CURRENT_DIR)
SUBMISSION_ROOT = resolve_submission_root(NOTEBOOK_DIR)
RESULT_DIR = ensure_directory(SUBMISSION_ROOT / 'results')
LOCAL_ASSET_DIR = NOTEBOOK_DIR / '_finetune_assets'
MODEL_CACHE_DIR = LOCAL_ASSET_DIR / 'model_cache'
TRAINING_OUTPUT_DIR = LOCAL_ASSET_DIR / 'training_outputs'
DATASET_PATH = ensure_local_dataset(NOTEBOOK_DIR, REMOTE_DATASET_URL)
TRAINING_INPUT_PATH = NOTEBOOK_DIR / 'ris_tax_training_100.csv'
TRAINING_FILE_TYPE = 'csv'
QUESTION_COLUMN = 'prompt'
ANSWER_COLUMN = 'answer'

BASE_MODEL_NAME = 'Qwen/Qwen3-0.6B'
# Slightly older but stronger instruction fallback:
# BASE_MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
CHECKPOINT_DIR = TRAINING_OUTPUT_DIR / 'qwen3_0_6b_lora'
RESULT_PATH = RESULT_DIR / 'finetune_qwen3_0_6b.csv'
FAILURE_LOG_PATH = RESULT_DIR / 'finetune_qwen3_0_6b_failures.json'
TRAINING_METADATA_PATH = CHECKPOINT_DIR / 'run_metadata.json'

USE_PEFT = True
FAST_LOCAL_TRAINING = True
ULTRAFAST_VALIDATION_MODE = True

MAX_TRAIN_ROWS = 12 if ULTRAFAST_VALIDATION_MODE else (60 if FAST_LOCAL_TRAINING else None)
MAX_SEQ_LENGTH = 128 if ULTRAFAST_VALIDATION_MODE else (192 if FAST_LOCAL_TRAINING else 384)
NUM_TRAIN_EPOCHS = 1 if FAST_LOCAL_TRAINING else 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1 if FAST_LOCAL_TRAINING else 4
WARMUP_RATIO = 0.05
LOGGING_STEPS = 5 if ULTRAFAST_VALIDATION_MODE else (20 if FAST_LOCAL_TRAINING else 5)
LORA_R = 2 if ULTRAFAST_VALIDATION_MODE else (4 if FAST_LOCAL_TRAINING else 8)
SAVE_STRATEGY = 'no' if FAST_LOCAL_TRAINING else 'epoch'
SAVE_TOTAL_LIMIT = 1 if FAST_LOCAL_TRAINING else 2
USE_GRADIENT_CHECKPOINTING = not FAST_LOCAL_TRAINING

GENERATION_BATCH_SIZE = 1
MAX_INPUT_TOKENS = 256 if ULTRAFAST_VALIDATION_MODE else 420
MAX_NEW_TOKENS = 64 if ULTRAFAST_VALIDATION_MODE else 96
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 40
DO_SAMPLE = True
REPETITION_PENALTY = 1.08
MAX_GENERATED_SENTENCES = 3
ROW_LIMIT = None
FINAL_EXPORT = False
ALLOW_REMOTE_MISMATCH_FOR_SMOKE_TEST = True
MAX_RETRIES = 2
MIN_ANSWER_LENGTH = 20
SEED = 42

random.seed(SEED)
set_seed(SEED)
ensure_directory(LOCAL_ASSET_DIR)
ensure_directory(MODEL_CACHE_DIR)
ensure_directory(TRAINING_OUTPUT_DIR)
ensure_directory(CHECKPOINT_DIR)

print('Current directory:', CURRENT_DIR)
print('Notebook directory:', NOTEBOOK_DIR)
print('Submission root:', SUBMISSION_ROOT)
print('Training data path:', TRAINING_INPUT_PATH)
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Result path:', RESULT_PATH)
print('Base model:', BASE_MODEL_NAME)


Notebook directory: D:\DS Applications\Vincent
Training data path: D:\DS Applications\Vincent\ris_tax_training_100.csv
Checkpoint dir: D:\DS Applications\Vincent\_finetune_assets\training_outputs\qwen3_0_6b_lora_merged
Result path: D:\DS Applications\Vincent\finetune_qwen3_0_6b_merged.csv
Base model: Qwen/Qwen3-0.6B


In [6]:
REQUIRED_COLUMNS = {'id', 'prompt'}
SYSTEM_PROMPT = (
    'Du bist ein hilfreicher Assistent fuer oesterreichisches Steuerrecht. '
    'Antworte auf Deutsch, kurz und praezise. '
    'Erfinde keine Fakten. Wenn du unsicher bist, sage das knapp.'
)


def normalize_answer(value) -> str:
    return ' '.join(str(value or '').strip().split())


def retry_call(fn, attempts: int = 2):
    last_error = None
    for _ in range(attempts):
        try:
            return fn()
        except Exception as exc:
            last_error = exc
    raise last_error


def read_remote_row_count() -> int | None:
    try:
        response = requests.get(REMOTE_DATASET_URL, timeout=30)
        response.raise_for_status()
        remote_frame = pd.read_csv(pd.io.common.StringIO(response.text), encoding='utf-8-sig')
        return len(remote_frame)
    except Exception as exc:
        print(f'Remote dataset check skipped: {exc}')
        return None


def load_benchmark_rows(dataset_path: Path, row_limit: int | None = None) -> list[dict]:
    if not dataset_path.exists():
        raise FileNotFoundError(f'Dataset not found: {dataset_path}')

    frame = pd.read_csv(dataset_path, encoding='utf-8-sig', dtype={'id': 'string', 'prompt': 'string'}, low_memory=False)
    missing_columns = REQUIRED_COLUMNS - set(frame.columns)
    if missing_columns:
        raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

    if frame['id'].duplicated().any():
        duplicate_ids = frame.loc[frame['id'].duplicated(), 'id'].astype(str).tolist()
        raise ValueError(f'Duplicate ids found in dataset: {duplicate_ids[:10]}')

    source_frame = frame.head(row_limit).copy() if row_limit is not None else frame.copy()
    rows = []
    for row in source_frame.itertuples(index=False):
        question = normalize_answer(row.prompt)
        if question:
            rows.append({'id': str(row.id), 'prompt': question})
    if not rows:
        raise ValueError('No usable benchmark rows were found.')
    return rows


def load_existing_submission(result_path: Path) -> dict[str, str]:
    if not result_path.exists():
        return {}
    frame = pd.read_csv(result_path, encoding='utf-8-sig', dtype={'id': 'string', 'answer': 'string'})
    if list(frame.columns) != ['id', 'answer']:
        raise ValueError('Existing submission file must have exactly the columns id,answer')
    if frame['id'].duplicated().any():
        raise ValueError('Existing submission contains duplicate ids.')
    return {str(row.id): normalize_answer(row.answer) for row in frame.itertuples(index=False) if normalize_answer(row.answer)}


def write_submission(ordered_rows: list[tuple[str, str]], result_path: Path):
    result_frame = pd.DataFrame(ordered_rows, columns=['id', 'answer'])
    result_frame.to_csv(result_path, index=False, encoding='utf-8-sig')


def validate_submission(benchmark_rows: list[dict], answers: dict[str, str], min_answer_length: int = 1) -> dict:
    expected_ids = [row['id'] for row in benchmark_rows]
    missing_ids = [row_id for row_id in expected_ids if row_id not in answers]
    extra_ids = [row_id for row_id in answers if row_id not in set(expected_ids)]
    short_answer_ids = [row_id for row_id in expected_ids if row_id in answers and len(normalize_answer(answers[row_id])) < min_answer_length]
    return {
        'row_count': len(answers),
        'missing_ids': missing_ids,
        'extra_ids': extra_ids,
        'short_answer_ids': short_answer_ids,
    }


full_benchmark_rows = load_benchmark_rows(DATASET_PATH)
remote_row_count = read_remote_row_count()
if remote_row_count is not None and len(full_benchmark_rows) != remote_row_count:
    message = f'Local dataset has {len(full_benchmark_rows)} rows but GitHub currently reports {remote_row_count}.'
    if FINAL_EXPORT or not ALLOW_REMOTE_MISMATCH_FOR_SMOKE_TEST:
        raise ValueError(message)
    print('Warning:', message)
else:
    print('Row-count check passed.')

benchmark_rows = full_benchmark_rows[:ROW_LIMIT] if ROW_LIMIT is not None else full_benchmark_rows
print('Benchmark rows loaded:', len(benchmark_rows))


Row-count check passed.
Benchmark rows loaded: 643


In [7]:
def block_official_testset(training_path: Path):
    forbidden_names = {'dataset_clean.csv', 'dataset_clean (1).csv'}
    if training_path.resolve() == DATASET_PATH.resolve() or training_path.name in forbidden_names:
        raise ValueError('The official benchmark looks like your training file. That is not allowed.')


def pick_device():
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def render_model_prompt(tokenizer, question: str, answer: str | None = None, add_generation_prompt: bool = False) -> str:
    cleaned_question = normalize_answer(question)
    cleaned_answer = normalize_answer(answer) if answer is not None else None
    if getattr(tokenizer, 'chat_template', None):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': cleaned_question},
        ]
        if cleaned_answer is not None:
            messages.append({'role': 'assistant', 'content': cleaned_answer})
            if BASE_MODEL_NAME.startswith('Qwen/Qwen3'):
                return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        if BASE_MODEL_NAME.startswith('Qwen/Qwen3'):
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False)
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)
    prompt = f'Instruction: {SYSTEM_PROMPT}\nQuestion: {cleaned_question}\nAnswer:'
    if cleaned_answer is None:
        return prompt
    return prompt + cleaned_answer


def load_training_examples(training_path: Path) -> list[dict]:
    if not training_path.exists():
        raise FileNotFoundError(f'Training data not found: {training_path}')
    block_official_testset(training_path)

    rows = []
    if TRAINING_FILE_TYPE == 'csv':
        with training_path.open('r', encoding='utf-8-sig', newline='') as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                question = normalize_answer(row.get(QUESTION_COLUMN) or '')
                answer = normalize_answer(row.get(ANSWER_COLUMN) or '')
                if question and answer:
                    rows.append({'prompt': question, 'answer': answer})
    elif TRAINING_FILE_TYPE == 'jsonl':
        with training_path.open('r', encoding='utf-8') as handle:
            for line in handle:
                payload = json.loads(line)
                question = normalize_answer(payload.get(QUESTION_COLUMN) or '')
                answer = normalize_answer(payload.get(ANSWER_COLUMN) or '')
                if question and answer:
                    rows.append({'prompt': question, 'answer': answer})
    else:
        raise ValueError(f'Unsupported training file type: {TRAINING_FILE_TYPE}')

    deduplicated = []
    seen = set()
    for row in rows:
        key = (row['prompt'], row['answer'])
        if key not in seen:
            seen.add(key)
            deduplicated.append(row)

    random.Random(SEED).shuffle(deduplicated)
    if MAX_TRAIN_ROWS is not None:
        deduplicated = deduplicated[:MAX_TRAIN_ROWS]
    if not deduplicated:
        raise ValueError('No usable training examples were found.')
    return deduplicated


def load_base_model_and_tokenizer(base_model_name: str):
    device = pick_device()
    if device.type == 'cpu':
        torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))

    tokenizer = AutoTokenizer.from_pretrained(base_model_name, cache_dir=str(MODEL_CACHE_DIR), use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {'cache_dir': str(MODEL_CACHE_DIR), 'low_cpu_mem_usage': True}
    if device.type in {'cuda', 'mps'}:
        model_kwargs['torch_dtype'] = 'auto'
    model = AutoModelForCausalLM.from_pretrained(base_model_name, **model_kwargs)
    model.to(device)
    model.config.use_cache = False
    return model, tokenizer, device


def build_training_dataset(rows: list[dict], tokenizer):
    encoded_rows = []
    skipped_rows = 0

    for row in rows:
        full_text = render_model_prompt(tokenizer, row['prompt'], row['answer'])
        prompt_only_text = render_model_prompt(tokenizer, row['prompt'], answer=None, add_generation_prompt=True)

        full_tokens = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH, add_special_tokens=False)
        prompt_tokens = tokenizer(prompt_only_text, truncation=True, max_length=MAX_SEQ_LENGTH, add_special_tokens=False)

        input_ids = full_tokens['input_ids']
        attention_mask = full_tokens['attention_mask']
        prompt_length = min(len(prompt_tokens['input_ids']), len(input_ids))

        labels = list(input_ids)
        labels[:prompt_length] = [-100] * prompt_length
        if all(label == -100 for label in labels):
            skipped_rows += 1
            continue

        encoded_rows.append({
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'length': len(input_ids),
        })

    if not encoded_rows:
        raise ValueError('No trainable examples were left after tokenization.')
    print('Prepared train rows:', len(encoded_rows), '| skipped after truncation:', skipped_rows)
    return Dataset.from_list(encoded_rows)


def collate_sft_batch(features: list[dict]):
    max_len = max(len(feature['input_ids']) for feature in features)
    input_ids = []
    attention_mask = []
    labels = []
    for feature in features:
        pad_len = max_len - len(feature['input_ids'])
        input_ids.append(feature['input_ids'] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(feature['attention_mask'] + [0] * pad_len)
        labels.append(feature['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
    }


def attach_lora_if_requested(model):
    if not USE_PEFT:
        return model
    lower_name = BASE_MODEL_NAME.lower()
    if 'gpt2' in lower_name:
        peft_config = LoraConfig(task_type='CAUSAL_LM', r=LORA_R, lora_alpha=16, lora_dropout=0.05, target_modules=['c_attn', 'c_proj'], fan_in_fan_out=True)
    else:
        target_modules = ['q_proj', 'v_proj'] if ULTRAFAST_VALIDATION_MODE else ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
        peft_config = LoraConfig(task_type='CAUSAL_LM', r=LORA_R, lora_alpha=16, lora_dropout=0.05, target_modules=target_modules)
    wrapped_model = get_peft_model(model, peft_config)
    if hasattr(wrapped_model, 'print_trainable_parameters'):
        wrapped_model.print_trainable_parameters()
    return wrapped_model


def load_checkpoint_for_generation(checkpoint_dir: Path, base_model_name: str, tokenizer):
    if not checkpoint_dir.exists() or not any(checkpoint_dir.iterdir()):
        raise FileNotFoundError(
            f'No fine-tuned checkpoint was found in {checkpoint_dir}. '
            'Run the training cell first and make sure it finishes with saved artifacts.'
        )

    base_model, _, device = load_base_model_and_tokenizer(base_model_name)
    if (checkpoint_dir / 'adapter_config.json').exists():
        model = PeftModel.from_pretrained(base_model, str(checkpoint_dir))
    elif (checkpoint_dir / 'config.json').exists():
        model_kwargs = {'cache_dir': str(MODEL_CACHE_DIR), 'low_cpu_mem_usage': True}
        if device.type in {'cuda', 'mps'}:
            model_kwargs['torch_dtype'] = 'auto'
        model = AutoModelForCausalLM.from_pretrained(str(checkpoint_dir), **model_kwargs)
        model.to(device)
    else:
        raise FileNotFoundError(
            f'Checkpoint folder {checkpoint_dir} exists, but it does not contain adapter_config.json or config.json. '
            'The training step likely did not save successfully.'
        )
    model.config.use_cache = True
    model.eval()
    return model, device


def clean_generated_answer(text: str) -> str:
    compact_text = str(text or '').replace('<think>', ' ').replace('</think>', ' ')
    compact_text = normalize_answer(compact_text)
    if not compact_text:
        return compact_text

    sentence_list = []
    for piece in re.split(r'[.!?]+', compact_text):
        piece = normalize_answer(piece)
        if not piece:
            continue
        if 'Gesetzesnummer-' in piece or piece.lower().endswith('.txt'):
            continue
        sentence_list.append(piece)

    unique_sentence_list = []
    seen_set = set()
    for sentence in sentence_list:
        key = sentence.lower()
        if key not in seen_set:
            seen_set.add(key)
            unique_sentence_list.append(sentence)

    cleaned_text = '. '.join(unique_sentence_list[:MAX_GENERATED_SENTENCES]).strip()
    cleaned_text = normalize_answer(cleaned_text)
    if cleaned_text and cleaned_text[-1] not in '.!?':
        cleaned_text += '.'
    return cleaned_text


def generate_batch(model, tokenizer, device, prompts: list[str]) -> list[str]:
    model_prompts = [render_model_prompt(tokenizer, prompt, answer=None, add_generation_prompt=True) for prompt in prompts]
    model_inputs = tokenizer(model_prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_INPUT_TOKENS)
    model_inputs = {key: value.to(device) for key, value in model_inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=DO_SAMPLE,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded_answers = []
    prompt_lengths = model_inputs['attention_mask'].sum(dim=1).tolist()
    for index, prompt_length in enumerate(prompt_lengths):
        answer_ids = generated_ids[index][int(prompt_length):]
        answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
        decoded_answers.append(clean_generated_answer(answer_text))
    return decoded_answers


In [8]:
training_examples = load_training_examples(TRAINING_INPUT_PATH)
model, tokenizer, device = load_base_model_and_tokenizer(BASE_MODEL_NAME)
model = attach_lora_if_requested(model)
tokenized_train_dataset = build_training_dataset(training_examples, tokenizer)

use_cpu_mode = not torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to='none',
    optim='adamw_torch',
    use_cpu=use_cpu_mode,
    bf16=False,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=torch.cuda.is_available(),
)

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train_dataset, data_collator=collate_sft_batch)
print('Trainer is ready on', device)
print('Training examples:', len(training_examples))
print('Tokenized train rows:', len(tokenized_train_dataset))


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 7979.62it/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 286,720 || all params: 596,336,640 || trainable%: 0.0481
Prepared train rows: 12 | skipped after truncation: 0
Trainer is ready on cuda
Training examples: 12
Tokenized train rows: 12


In [9]:
try:
    train_result = trainer.train()
    model.config.use_cache = True
    trainer.save_model(str(CHECKPOINT_DIR))
    tokenizer.save_pretrained(str(CHECKPOINT_DIR))
    training_metadata = {
        'base_model_name': BASE_MODEL_NAME,
        'training_input_path': str(TRAINING_INPUT_PATH),
        'checkpoint_dir': str(CHECKPOINT_DIR),
        'use_peft': USE_PEFT,
        'num_train_examples': len(training_examples),
        'num_train_epochs': NUM_TRAIN_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'max_seq_length': MAX_SEQ_LENGTH,
        'generation_defaults': {
            'max_input_tokens': MAX_INPUT_TOKENS,
            'max_new_tokens': MAX_NEW_TOKENS,
            'temperature': TEMPERATURE,
            'top_p': TOP_P,
            'top_k': TOP_K,
            'do_sample': DO_SAMPLE,
            'repetition_penalty': REPETITION_PENALTY,
        },
        'trainer_metrics': getattr(train_result, 'metrics', {}),
    }
    TRAINING_METADATA_PATH.write_text(json.dumps(training_metadata, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Training finished and artifacts were saved.')
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower():
        raise RuntimeError('Training ran out of memory. Lower MAX_SEQ_LENGTH, reduce the batch size, or move this notebook to Colab/Kaggle.') from exc
    raise


Step,Training Loss
5,2.616730
10,2.186332


Training finished and artifacts were saved.


In [10]:
generation_model, generation_device = load_checkpoint_for_generation(CHECKPOINT_DIR, BASE_MODEL_NAME, tokenizer)
existing_answers = load_existing_submission(RESULT_PATH)
pending_rows = [row for row in benchmark_rows if row['id'] not in existing_answers]

timings = []
failures = {}

for batch_start in tqdm(range(0, len(pending_rows), GENERATION_BATCH_SIZE), desc='Fine-tuned inference'):
    batch_rows = pending_rows[batch_start: batch_start + GENERATION_BATCH_SIZE]
    prompts = [row['prompt'] for row in batch_rows]
    started = time.perf_counter()
    try:
        batch_answers = retry_call(lambda: generate_batch(generation_model, tokenizer, generation_device, prompts), attempts=MAX_RETRIES)
    except Exception as batch_exc:
        print(f'Fine-tuned batch starting at row {batch_start} failed. Falling back to one-by-one mode. Error: {batch_exc}')
        batch_answers = []
        for row in batch_rows:
            try:
                answer = retry_call(
                    lambda row=row: generate_batch(generation_model, tokenizer, generation_device, [row['prompt']])[0],
                    attempts=MAX_RETRIES,
                )
                batch_answers.append(answer)
            except Exception as row_exc:
                failures[row['id']] = str(row_exc)
                batch_answers.append(None)
    timings.append(time.perf_counter() - started)

    for row, answer in zip(batch_rows, batch_answers):
        if answer:
            cleaned = clean_generated_answer(answer)
            if cleaned:
                existing_answers[row['id']] = cleaned
            else:
                failures[row['id']] = 'The fine-tuned model returned an empty answer after cleaning.'
        else:
            failures.setdefault(row['id'], 'No answer was generated for this row.')

    ordered_rows = [(row['id'], existing_answers[row['id']]) for row in benchmark_rows if row['id'] in existing_answers]
    write_submission(ordered_rows, RESULT_PATH)
    FAILURE_LOG_PATH.write_text(json.dumps(failures, indent=2, ensure_ascii=False), encoding='utf-8')

print('Fine-tuned inference loop finished.')


Fine-tuned inference: 100%|██████████| 643/643 [43:48<00:00,  4.09s/it]

Fine-tuned inference loop finished.


In [11]:
validation = validate_submission(benchmark_rows, existing_answers, min_answer_length=MIN_ANSWER_LENGTH)
summary = {
    'rows_written': validation['row_count'],
    'failed_rows': len(failures),
    'average_batch_seconds': round(statistics.mean(timings), 3) if timings else 0.0,
    'short_answer_count': len(validation['short_answer_ids']),
    'result_path': str(RESULT_PATH),
    'checkpoint_dir': str(CHECKPOINT_DIR),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('\nShort answers worth reviewing:', validation['short_answer_ids'][:10])
pd.read_csv(RESULT_PATH, encoding='utf-8-sig').head(5)


{
  "rows_written": 643,
  "failed_rows": 0,
  "average_batch_seconds": 4.085,
  "short_answer_count": 1,
  "result_path": "D:\\DS Applications\\Vincent\\finetune_qwen3_0_6b_merged.csv",
  "checkpoint_dir": "D:\\DS Applications\\Vincent\\_finetune_assets\\training_outputs\\qwen3_0_6b_lora_merged"
}

Short answers worth reviewing: ['VAT-INTL-049']


,id,answer
0,CORP-TAX-001,Die steuerliche Bemessungsgrundlage für die Kö...
1,CORP-TAX-002,Wenn eine Körperschaft ein zinsloses Darlehen ...
2,CORP-TAX-003,Die verpflichteten Körperschaften sind die Gew...
3,CORP-TAX-004,- **(a)** Bei der Tochtergesellschaft: Die Div...
4,CORP-TAX-005,"Die Unterscheidung liegt darin, dass eine offe..."


## Result

Die Fine-Tune-CSV wird im Modellordner als `finetune_qwen3_0_6b.csv` gespeichert.
Der Checkpoint liegt im lokalen Unterordner `_finetune_assets/training_outputs/qwen3_0_6b_lora`.
